# Inspecting a night's products

This notebook opens the folder that `mastpipe quicklook` wrote and looks inside: the products
as the pipeline's own objects, their provenance, the drift that was applied, the kept
versions, the combined spectra, the reference solution, the QC files and the log.

To keep it self-contained it first **makes** a night: a simulated survey night (45 fibers)
is reduced with `identify` and `quicklook`, which takes a couple of minutes. To use it on your
own data, skip the next cell and point `OUT` at your output folder.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mastpipe.config.loader import build_effective
from mastpipe.instruments.registry import load_instrument, load_layout, load_line_table
from mastpipe.pipeline.identify import run_identify
from mastpipe.pipeline.runner import RunOptions, run_quicklook
from mastpipe.sim.simulator import simulate_night

work = Path(tempfile.mkdtemp(prefix="mastpipe-products-"))
NIGHT, OUT = work / "night", work / "out"

effective = build_effective(load_instrument("highspec"), "Mg", "main", layout="survey45")
truth = simulate_night(effective, load_layout("highspec", "survey45"), load_line_table("highspec", "Mg"),
                       NIGHT, seed=0, n_science=4, n_cal_arc=2)
frames = truth.frames.set_index("file")
full_arc = NIGHT / frames.index[frames.arc_kind == "full"][0]

# the first reference for this grating and layout, from two anchors (here: from the truth)
bright = np.where(truth.lines_on_chip & (truth.line_logI > 2.0))[0]
anchors = [(float(truth.lines_nm[i]), float(truth.line_pixels[0][i])) for i in (bright[0], bright[-1])]
reference, ref_path = run_identify(full_arc, effective, "survey45", OUT, anchors=anchors, anchor_trace=0)
print("reference:", reference.identifier, "-", len(reference.solutions), "fibers")

summary = run_quicklook(NIGHT, OUT, RunOptions(instrument="highspec", layout="survey45"))
print(f"{len(summary.products)} products, {len(summary.skipped)} skipped, {len(summary.failed)} failed")

## The folder

```text
OUT/
  calib/   masters, trace sets, reference solutions, index.json
  spec/    one *_spec.fits per science exposure, one *_comb_<target>.fits per target
  qa/      one JSON per trace set, reference, drift arc and science frame
  logs/    one log per run
```

In [ ]:
for sub in ("calib", "spec", "qa", "logs"):
    names = sorted(p.name for p in (OUT / sub).iterdir())
    print(f"{sub}/ ({len(names)})")
    for n in names[:6]:
        print("   ", n)
    if len(names) > 6:
        print("    …")

## The calibration folder

`calib/` holds what the run built or reused: the masters, the night's trace set and the
reference solution, all indexed in `index.json` with the validity keys the library matches
on (camera, binning, readout mode, temperature, time). Everything there is a FITS file the
`io.products` readers give back as the pipeline's own objects.

In [ ]:
from mastpipe.io.products import read_master, read_traceset
from mastpipe.pipeline.callib import CalibLibrary

library = CalibLibrary(OUT / "calib")
pd.DataFrame(library.list())[["kind", "path", "configuration", "mid_time", "n_frames"]]

In [ ]:
masters = {}
for kind in ("bias", "dark", "flat", "background"):
    master_file = sorted((OUT / "calib").glob(f"*_{kind}.fits"))[-1]
    masters[kind], _ = read_master(master_file)

fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, (kind, master) in zip(axes, masters.items(), strict=True):
    lo, hi = np.nanpercentile(master.data, [2, 99.5])
    im = ax.imshow(master.data, origin="lower", cmap="gray", vmin=lo, vmax=hi, aspect="auto")
    ax.set(title=f"master {kind} ({master.n_frames} frames)", xlabel="column", ylabel="row")
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
plt.tight_layout()
plt.show()

The trace set found on the flat: 45 traces matched to the `survey45` layout and keyed by its
trace ids, drawn over the flat they were found on. The arc (calibration) fibers are the five
that `identify` and the drift fits use; science and sky fibers alternate between them.

In [ ]:
traceset, _ = read_traceset(sorted((OUT / "calib").glob("*_traces.fits"))[-1])
role_color = {"arc": "tab:green", "science": "tab:red", "sky": "tab:blue"}
flat = masters["flat"].data
x = np.arange(flat.shape[1])
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 1]}, sharey=True)
lo, hi = np.nanpercentile(flat, [5, 99.5])
axes[0].imshow(flat, origin="lower", cmap="gray", vmin=lo, vmax=hi, aspect="auto")
for t in traceset:
    axes[0].plot(x, t.y(x), color=role_color.get(t.role, "0.5"), lw=0.6)
axes[0].set(xlabel="column", ylabel="row", title=f"{len(traceset)} traces on the master flat (green arc, red science, blue sky); missing {traceset.missing}, extra {traceset.extra}")
profile = np.nanmedian(flat[:, 700:900], axis=1)
axes[1].plot(profile, np.arange(flat.shape[0]), color="0.3", lw=0.8)
for t in traceset:
    axes[1].axhline(t.y_center, color=role_color.get(t.role, "0.5"), lw=0.5, alpha=0.7)
axes[1].set(xlabel="flat level (ADU)", title="collapsed profile")
plt.tight_layout()
plt.show()

## A science product

`read_spectra` returns the exposure's spectra, its provenance and the drift model that was
applied. The statuses say what the wavelengths rest on.

In [ ]:
from mastpipe.io.products import read_spectra

spec_files = sorted((OUT / "spec").glob("*_spec.fits"))
es, prov, drift = read_spectra(spec_files[-1])
print(spec_files[-1].name)
print("wavelength:", es.wavelength_status, "| flux:", es.flux_status, "| telluric:", es.telluric_status)
print("reference:", es.reference_id, "| drift arcs:", list(es.drift_arcs))
print("pipeline", prov.pipeline_version, "| configuration hash", prov.config_hash)
print("calibrations:", prov.calibrations)
print("steps:", *prov.steps, sep="\n  ")

In [ ]:
def summary_table(es):
    rows = []
    for s in es.spectra:
        good = ~s.mask
        snr = float(np.nanmedian(s.flux[good] / s.error[good])) if good.any() else np.nan
        rows.append(dict(fiber=s.fiber, role=s.role, telescope=s.telescope, target=s.target,
                         median_snr=round(snr, 1), masked=f"{s.mask.mean():.1%}",
                         wavelength=s.wavelength is not None, versions=",".join(sorted(s.versions)) or "-"))
    return pd.DataFrame(rows)

summary_table(es).head(12)

The same table as a picture, for all 45 fibers: the median signal-to-noise of each fiber's raw
extraction and the fraction of its pixels that are masked. Calibration fibers carry no light on a
science frame, so their signal-to-noise sits near zero; a science or sky fiber that falls out of
line with its neighbours is the one to look at.

In [ ]:
table = summary_table(es)
table["masked_fraction"] = [s.mask.mean() for s in es.spectra]
colors = [role_color.get(r, "0.5") for r in table.role]
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].bar(table.fiber, table.median_snr, color=colors)
axes[0].set(ylabel="median S/N per pixel", title=f"{es.meta.source}: every fiber's raw extraction (green arc, red science, blue sky)")
axes[1].bar(table.fiber, table.masked_fraction * 100, color=colors)
axes[1].set(xlabel="fiber (layout trace id)", ylabel="masked (%)")
plt.tight_layout()
plt.show()

## Plotting a fiber

The raw extraction, the sky that was subtracted and the sky-subtracted version live side by
side in the same row; nothing was overwritten.

In [ ]:
s = next(s for s in es.spectra if s.role == "science" and "skysub" in s.versions)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(s.wavelength, s.flux, lw=0.7, color="0.5", label="raw extraction")
axes[0].plot(s.wavelength, s.sky, lw=0.7, color="tab:blue", label="sky, scaled and resampled")
axes[1].plot(s.wavelength, s.versions["skysub"].flux, lw=0.7, color="tab:red", label="sky-subtracted")
axes[1].fill_between(s.wavelength, s.versions["skysub"].flux - s.versions["skysub"].error,
                     s.versions["skysub"].flux + s.versions["skysub"].error, color="tab:red", alpha=0.2)
axes[0].set(title=f"fiber {s.fiber}, telescope {s.telescope}, target {s.target}", ylabel="ADU")
axes[1].set(xlabel="wavelength (nm)", ylabel="ADU")
for ax in axes:
    ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

All twenty sky-subtracted science fibers of the exposure, offset and labelled by telescope. On
a real night this is the quickest check that every telescope delivered light and that no fiber's
sky subtraction went wrong.

In [ ]:
science_fibers = [s for s in es.spectra if s.role == "science" and "skysub" in s.versions]
fig, ax = plt.subplots(figsize=(12, 9))
step = 1.1 * np.nanpercentile([s.versions["skysub"].flux for s in science_fibers], 99.5)
for k, s in enumerate(science_fibers):
    ax.plot(s.wavelength, s.versions["skysub"].flux + k * step, lw=0.5, color="tab:red")
    ax.text(s.wavelength[-1] + 0.1, k * step, f"{s.fiber} {s.telescope}", va="center", fontsize=8)
ax.set(xlabel="wavelength (nm)", ylabel="ADU + offset", yticks=[], title=f"{len(science_fibers)} science fibers, sky-subtracted")
plt.show()

## The drift that was applied

The `DRIFT` table holds, per calibration fiber, the shift and stretch measured against the
reference, and the smoothed model across the detector that the science fibers used.

In [ ]:
print("arc mid-time", drift.arc_mid_time, "| usable", drift.usable, "| scatter", f"{drift.scatter_px:.4f} px")
display(drift.per_fiber)
y = np.linspace(0, 400, 50)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(drift.per_fiber.y_center, drift.per_fiber.shift_px, "o", label="calibration fibers")
ax.plot(y, drift.shift_at(y), "-", color="0.4", label="smoothed model")
planted = frames.loc[es.meta.source, "shift_px"]
ax.axhline(planted, color="tab:green", ls="--", label=f"planted at the science frame's time ({planted:.3f} px)")
ax.set(xlabel="trace centre (row)", ylabel="shift (px)", title="drift transferred to the science fibers")
ax.legend(fontsize=8)
plt.show()

The gap between the dashed line and the model is the price of the quick-look's
`nearest_before` policy: the last science frame of the night is reduced with the arc taken
before it, while the spectrograph kept drifting; no later arc exists to interpolate towards.
At the observatory the next calibration-fiber arc closes the gap for the frames that follow,
and a manual re-reduction with `--time-policy interpolate` uses the arcs on both sides where
they exist. The drift model's own scatter across the five fibers is the number to watch.

## Combined spectra

One file per target and exposure: the contributors, their weights, and every version the
contributors share, combined separately. (The simulator writes a single `OBJECT` card per
frame naming all four of its targets, so `quicklook` sees one target here and combines all
twenty science fibers; on real frames each telescope's target comes from its own pointing.)

In [ ]:
from mastpipe.io.products import read_combined

comb_files = sorted((OUT / "spec").glob("*_comb_*.fits"))
cs, meta, cprov = read_combined(comb_files[-1])
print(comb_files[-1].name, "| target", cs.target, "| grouping", cs.grouping, "| versions", list(cs.versions))
display(cs.contributors)
if len(cs.excluded):
    display(cs.excluded)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.4), gridspec_kw={"width_ratios": [3, 1.2]})
axes[0].plot(cs.wavelength, cs.versions["raw"].flux, lw=0.7, color="0.5", label="raw")
axes[0].plot(cs.wavelength, cs.versions["skysub"].flux, lw=0.7, color="tab:red", label="sky-subtracted")
axes[0].fill_between(cs.wavelength, cs.versions["skysub"].flux - cs.versions["skysub"].error,
                     cs.versions["skysub"].flux + cs.versions["skysub"].error, color="tab:red", alpha=0.25)
axes[0].set(xlabel="wavelength (nm)", ylabel="ADU", title=f"combined: {cs.target} ({len(cs.contributors)} contributors)")
axes[0].legend()
axes[1].bar(cs.contributors.fiber, cs.contributors.weight, color="tab:gray")
axes[1].set(xlabel="fiber", ylabel="weight", title="contributor weights")
plt.tight_layout()
plt.show()

## The reference solution

The calibration library finds the reference valid at any time; each fiber's solution carries
its identified lines, residuals and the range it was fitted over.

In [ ]:
from mastpipe.pipeline.callib import CalibLibrary

library = CalibLibrary(OUT / "calib")
ref = library.find_reference("highspec", "Mg", "survey45", at_time=es.meta.mid_time)
print(ref.identifier, "| valid from", ref.valid_from, "|", len(ref.solutions), "fibers")
table = pd.DataFrame({"fiber": list(ref.solutions),
                      "lines": [s.n_lines for s in ref.solutions.values()],
                      "rms_px": [s.rms_px for s in ref.solutions.values()],
                      "nm_from": [s.anchor_nm[0] for s in ref.solutions.values()],
                      "nm_to": [s.anchor_nm[1] for s in ref.solutions.values()]})
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(table.fiber, table.rms_px, color="tab:blue"); axes[0].set(xlabel="fiber", ylabel="rms (px)", title="solution rms per fiber")
axes[1].bar(table.fiber, table.lines, color="tab:orange"); axes[1].set(xlabel="fiber", ylabel="lines used", title="identified lines per fiber")
plt.tight_layout()
plt.show()
table.describe().loc[["min", "50%", "max"], ["lines", "rms_px"]]

## The QC files

One JSON per product with the numbers behind the summaries, for a dashboard to read.

In [ ]:
from mastpipe.io.products import read_qc

qc_file = next((OUT / "qa").glob(spec_files[-1].stem.replace("_spec", "") + "*spec*.json"), None) or sorted((OUT / "qa").glob("*spec*.json"))[-1]
qc = read_qc(qc_file)
print(qc_file.name, "| stage", qc.stage)
print(sorted(qc.values))
print("drift arcs:", qc.values.get("drift_arcs"), "weights:", qc.values.get("drift_weights"))
print("fibers without a wavelength:", qc.values.get("wavelength_skipped"))
print("sky subtraction skipped:", qc.values.get("sky"))

The QC numbers are enough for a dashboard without opening the FITS product: here the drift
shift that each fiber received and the median signal-to-noise, straight from the JSON.

In [ ]:
qc_fibers = np.array(qc.values["fibers"])
qc_roles = qc.values["roles"]
fig, axes = plt.subplots(1, 2, figsize=(13, 3))
axes[0].plot(qc_fibers, qc.values["drift_shift_px"], ".-", color="0.3")
axes[0].set(xlabel="fiber", ylabel="shift applied (px)", title=f"drift transferred to each fiber (scatter {qc.values['drift_scatter_px']:.4f} px)")
axes[1].bar(qc_fibers, qc.values["median_snr"], color=[role_color.get(r, "0.5") for r in qc_roles])
axes[1].set(xlabel="fiber", ylabel="median S/N", title="median S/N from the QC file")
plt.tight_layout()
plt.show()

## The log

In [ ]:
log = sorted((OUT / "logs").glob("quicklook_*.log"))[-1]
print(log.name)
print(*log.read_text(encoding="utf-8").splitlines()[-12:], sep="\n")

## Against the truth

Because this night was simulated, the wavelengths can be checked against the simulator's law.
On a real night the equivalent check is the fiber-to-fiber agreement in `fiber_agreement.csv`.

In [ ]:
from mastpipe.sim.simulator import undrift_x
from mastpipe.steps.geometry import grating_geometry
from mastpipe.steps.wavemodel import GratingModel

geom = grating_geometry(effective.geometry, effective.detector.pixel_size_um)
n_pix = len(es.spectra[0].pixel)
truth_model = GratingModel(x_c=truth.x_c, B=truth.B, alpha=geom.alpha_rad, poly=None, n_pix=n_pix, geom=geom)
row = frames.loc[es.meta.source]
curves, worst = {}, {}
for s in es.spectra:
    if s.wavelength is None or s.role != "science":
        continue
    px = s.pixel.astype(float)
    lam_true = truth_model.wavelength(undrift_x(px, row.shift_px, row.stretch, n_pix))
    curves[s.fiber] = (s.wavelength - lam_true) / truth_model.dispersion(px)
    worst[s.fiber] = float(np.nanmax(np.abs(curves[s.fiber])))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), gridspec_kw={"width_ratios": [2, 1]})
for fiber, curve in curves.items():
    axes[0].plot(curve, lw=0.7, label=f"fiber {fiber}")
axes[0].axhline(0, color="0.3", lw=0.5)
axes[0].set(xlabel="pixel", ylabel="(pipeline − truth) / dispersion", title="wavelength error along every science fiber")
axes[1].bar(list(worst), list(worst.values()), color="tab:red")
axes[1].axhline(0.3, color="0.3", ls="--", label="test tolerance 0.3")
axes[1].set(xlabel="fiber", ylabel="worst |error| (dispersions)", title="worst error per fiber")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"{len(worst)} science fibers: worst wavelength error {max(worst.values()):.3f} dispersions (test tolerance 0.3)")